In [1]:
import folium
import ast
from IPython.display import display
import mercantile
from shapely import geometry, wkt

import sparql_virtualizer
import rdflib

g = rdflib.Graph()


query = """
PREFIX ogc: <http://www.ogc.org/>
PREFIX ine: <http://lod.ine.es/voc/cubes/vocabulary#>
PREFIX sdmx-measure: <http://purl.org/linked-data/sdmx/2009/measure#>
PREFIX sdmx-dimension: <http://purl.org/linked-data/sdmx/2009/dimension#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>
PREFIX ex: <http://example.org/function/>
PREFIX qb: <http://purl.org/linked-data/cube#>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX geolinkeddata: <http://geo.linkeddata.es/ontology/> 

    SELECT ?t ?gt WHERE {
        ?g a ogc:administrativeunit ;
            ogc:nameunit "Santiago de Compostela" ;
            ogc:nationallevelname "Municipio" ;
            ogc:country "ES" ;
            geo:hasGeometry ?gg .

        ?y a ogc:namedplace ;
            ogc:etiqueta ?ny ;
            ogc:tipo "Barrio" ;
            geo:hasGeometry ?gy .
        FILTER(geof:sfContains(?gg, ?gy))

        ?t a ogc:copernicus_wcs ;
            ogc:coverage "NATURAL-COLOR" ;
            geo:hasGeometry ?gt .
        FILTER(geof:sfIntersects(?gt, ?gy))
    } 
"""

def parse_geometry_literal(raw_geom):
    raw_geom = str(raw_geom).strip()
    if raw_geom.startswith("{"):
        return ast.literal_eval(raw_geom)
    if raw_geom.startswith("<"):
        raw_geom = raw_geom.partition(">")[2].strip()
    return geometry.mapping(wkt.loads(raw_geom))

qres = g.query(query)

# 1. Configuración base
# Usamos un mapa base oscuro para que los bordes de colores resalten más
m = folium.Map(location=[40.41, -3.70], zoom_start=6, tiles='CartoDB positron')
puntos_para_encuadre = []

# Paleta de colores más brillantes para que destaquen sobre fondo oscuro y con bordes
colores_paleta = ['#3498db', '#e74c3c', '#2ecc71', '#f1c40f', '#9b59b6', '#e67e22']

for row in qres:
    # Iteramos sobre la fila de 2 en 2
    for i in range(0, len(row), 2):
        try:
            uri = str(row[i])
            raw_geom = row[i+1]
            geom_dict = parse_geometry_literal(raw_geom)
            
            # Selección de color
            color_index = (i // 2) % len(colores_paleta)
            color_actual = colores_paleta[color_index]
            
            # CONFIGURACIÓN DE ESTILO PARA EVITAR OPACIDAD
            # Usamos una opacidad de relleno MUY baja (0.1 o 0.2)
            # Pero una opacidad de borde ALTA (1.0) y un peso de línea mayor
            def style_function(feature, color=color_actual):
                return {
                    'fillColor': color,   # Color de relleno
                    'fillOpacity': 0.01,   # MUY TRANSPARENTE por dentro (15%)
                    'color': color,       # Color del borde
                    'opacity': 1.0,        # BORDE TOTALMENTE OPACO
                    'weight': 3,          # BORDE GRUESO para que se vea bien el contorno
                }
            
            # Añadir al mapa con el estilo modificado
            folium.GeoJson(
                geom_dict,
                tooltip=f"<b>Variable {i//2 + 1}:</b> {uri}",
                style_function=style_function
            ).add_to(m)
            
            # Recolectar coordenadas para auto-zoom
            if 'coordinates' in geom_dict:
                coords = geom_dict['coordinates']
                if geom_dict['type'] == 'Polygon':
                    # Manejo de Polígonos y MultiPolígonos si fuera necesario
                    p = coords[0][0] # Primer punto del primer anillo
                    puntos_para_encuadre.append([p[1], p[0]])
                elif geom_dict['type'] == 'Point':
                    puntos_para_encuadre.append([coords[1], coords[0]])
                    
        except Exception:
            # Capturar errores de parsing de geometría o valores nulos
            continue

# 2. Ajuste de vista
if puntos_para_encuadre:
    m.fit_bounds(puntos_para_encuadre)

display(m)

https://api-features.ign.es/collections/administrativeunit/items?f=json&limit=10000&country=ES&nameunit=Santiago+de+Compostela&nationallevelname=Municipio
https://api-features.ign.es//collections/namedplace/items?f=json&limit=10000&bbox=-8.632011636%2C42.824144224%2C-8.389642519%2C42.989630989000005&tipo=Barrio


In [5]:
import ast
import threading
import time
import rdflib
import sparql_virtualizer
from IPython.display import display
from ipyleaflet import GeoJSON, LayersControl, Map, basemaps
from ipywidgets import Output, VBox, Button  # ← añadir Button

g = rdflib.Graph()
status = Output()
last_update_at = 0.0
update_in_progress = False
pending_bounds = None

def bounds_to_wkt(bounds):
    (south, west), (north, east) = bounds
    return (
        f"POLYGON(({west} {south}, {east} {south}, {east} {north}, "
        f"{west} {north}, {west} {south}))"
    )

def query_features(bounds):
    bbox_wkt = bounds_to_wkt(bounds)
    query = f"""
PREFIX ogc: <http://www.ogc.org/>
PREFIX ine: <http://lod.ine.es/voc/cubes/vocabulary#>
PREFIX sdmx-measure: <http://purl.org/linked-data/sdmx/2009/measure#>
PREFIX sdmx-dimension: <http://purl.org/linked-data/sdmx/2009/dimension#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>
PREFIX ex: <http://example.org/function/>
PREFIX qb: <http://purl.org/linked-data/cube#>
PREFIX dbo: <http://dbpedia.org/ontology/>

    SELECT ?x ?gx WHERE {{
        ?x a dbo:RailwayStation ;
            geo:hasGeometry ?gx .
        FILTER (geof:sfWithin(?gx, \"{bbox_wkt}\"^^geo:wktLiteral))
    }}
"""
    features = []
    for uri, raw_geom in g.query(query):
        raw_geom = str(raw_geom)
        if not raw_geom.startswith("{"):
            continue
        try:
            geom = ast.literal_eval(raw_geom)
        except (SyntaxError, ValueError):
            continue
        features.append({
            "type": "Feature",
            "geometry": geom,
            "properties": {"uri": str(uri)},
        })
    return {"type": "FeatureCollection", "features": features}

geojson_layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={
        "color": "#e74c3c",
        "weight": 3,
        "opacity": 1.0,
        "fillOpacity": 0.05,
    },
    hover_style={"fillOpacity": 0.2},
    name="Railway stations",
)

m = Map(
    center=(40.41, -3.70),
    zoom=6,
    basemap=basemaps.CartoDB.Positron,
    scroll_wheel_zoom=True,
)
m.add_layer(geojson_layer)
m.add_control(LayersControl())

def refresh(bounds):
    global last_update_at, update_in_progress, pending_bounds
    if update_in_progress:
        pending_bounds = bounds
        return
    update_in_progress = True
    try:
        with status:
            status.clear_output(wait=True)
            print("Querying current bounding box...")
        data = query_features(bounds)
        geojson_layer.data = data
        last_update_at = time.monotonic()
        with status:
            status.clear_output(wait=True)
            print(f"Loaded {len(data['features'])} features for the current view.")
    except Exception as exc:
        with status:
            status.clear_output(wait=True)
            print(f"Error updating map: {exc}")
    finally:
        update_in_progress = False
        if pending_bounds is not None:
            next_bounds = pending_bounds
            pending_bounds = None
            refresh(next_bounds)

# ── Botón manual ──────────────────────────────────────────────
btn = Button(
    description="🔍 Buscar en esta zona",
    button_style="primary",
    layout={"width": "200px"},
)

def on_btn_click(_):
    bounds = m.bounds
    if bounds:
        print("Botón pulsado, actualizando mapa...")
        threading.Thread(target=refresh, args=(bounds,), daemon=True).start()

btn.on_click(on_btn_click)
# ─────────────────────────────────────────────────────────────

# ← ya NO hay m.observe(on_bounds_change, ...)

display(VBox([btn, m, status]))